In [ ]:
# X2DDR — pulls the code from GitHub so a run is reproducible from a commit SHA
import os, subprocess, sys, shutil, time
REPO = "https://github.com/Ahmadrezanourozii/Automatic-detection-of-diabetic-retinopathy-and-grading-of-diabetic-macular-edema-using-CNN.git"
COMMIT = "46f1bddad36e5ba74febdc21ef3e89b85c8e52b0"
WORK = "/kaggle/working/repo"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", REPO, WORK], check=True)
if COMMIT and COMMIT != "HEAD":
    subprocess.run(["git", "-C", WORK, "checkout", "--quiet", COMMIT], check=True)
sha = subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"]).decode().strip()
print("CODE COMMIT", sha)
print(subprocess.check_output(["git", "-C", WORK, "log", "-1", "--pretty=%s"]).decode().strip())


In [ ]:
import os
for d in sorted(os.listdir("/kaggle/input")):
    n = sum(len(f) for _, _, f in os.walk(f"/kaggle/input/{d}"))
    print(f"{d:55s} {n:7d} files")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


In [ ]:
# The pool does not always honour the pinned accelerator. A P100 is sm_60 and the
# preinstalled torch cu128 build ships sm_70+ kernels only, so every CUDA call fails.
# Rather than lose the run, install a torch that supports this device (ISSUES.md §9).
import subprocess, sys, torch
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU {name}  sm_{cap[0]}{cap[1]}  torch {torch.__version__}")
ok = True
try:
    (torch.zeros(8, 8, device="cuda") @ torch.zeros(8, 8, device="cuda")).sum().item()
    print("kernels execute fine on this device")
except Exception as e:
    ok = False
    print("UNUSABLE:", e)
if not ok:
    print("installing a torch build that supports this GPU ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=False)
    print("installed -- src/train.py runs in a subprocess so it picks up the new build")


In [ ]:
import subprocess, sys, os, glob, shutil, json
RUN_ID = "X2DDR"
OUT = f"/kaggle/working/{RUN_ID}"
os.makedirs(OUT, exist_ok=True)
SPEC = {'E08': ['E08'], 'E11FULL': ['E11', 'E19E11C'], 'E09': ['E09'], 'E10': ['E10'], 'E20CORAL': ['E20CORAL'], 'E22CORN': ['E22CORN'], 'E15LPFT': ['E15LPFT'], 'E14MAC': ['E14MAC'], 'E17NAT': ['E17NAT'], 'E21CNXA': ['E21CNXA']}
CORPORA = ['DDR', 'APTOS']
summary = {}
for name, srcdirs in SPEC.items():
    dst = os.path.join(OUT, name)
    os.makedirs(dst, exist_ok=True)
    found = {}
    for root, _, files in os.walk("/kaggle/input"):
        if os.path.basename(root) in srcdirs:
            for fn in files:
                if fn.startswith("best_") and fn.endswith(".pt"):
                    if fn in found:
                        raise SystemExit(f"{name}: {fn} supplied twice ({found[fn]} and {root})")
                    found[fn] = os.path.join(root, fn)
    print(f"{name}: {len(found)} fold weights from {srcdirs}: {sorted(found)}", flush=True)
    if not found:
        summary[name] = "NO WEIGHTS FOUND"
        continue
    for fn, src in found.items():
        shutil.copy2(src, os.path.join(dst, fn))
    for CORPUS in CORPORA:
        tag = CORPUS.lower()
        cmd = [sys.executable, "-u", "/kaggle/working/repo/src/eval_external.py",
               "--run", dst, "--datasets", "/kaggle/input", "--corpus", CORPUS,
               "--cache", f"/kaggle/temp/cache_ext_{tag}", "--tta"]
        print(" ".join(cmd), flush=True)
        with open(os.path.join(dst, f"external_{tag}.log"), "w") as f:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1, cwd="/kaggle/working/repo")
            for line in p.stdout:
                print(line, end="", flush=True); f.write(line); f.flush()
            p.wait()
        summary[f"{name}/{CORPUS}"] = p.returncode
    for p_ in glob.glob(os.path.join(dst, "best_*.pt")):
        os.remove(p_)        # they belong to the source runs; do not duplicate them
json.dump(summary, open(os.path.join(OUT, "multi_summary.json"), "w"), indent=1)
print(summary)
